# The full Transformer in PyTorch

Study notes that accompany [What is a Transformer? A visual guide, told with one T-cell receptor](https://sys0507.github.io/notebook/2026/attention-explained-with-t-cell-receptors/), Section 10.

The original encoder–decoder from *Attention Is All You Need* (Vaswani et al., 2017), built module by module: Post-LN, ReLU feed-forward, sinusoidal positions. Requires only `torch`; the training demo at the end takes a few minutes on a CPU.

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. Positional encoding

In [2]:
# 1. Positional encoding ---------------------------------------------------------------
class PositionalEncoding(nn.Module):
    """PE(pos, 2i) = sin(pos / 10000^(2i/d_model)),  PE(pos, 2i+1) = cos(same angle)."""
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(max_len).unsqueeze(1)                       # (max_len, 1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)                       # even dimensions
        pe[:, 1::2] = torch.cos(position * div_term)                       # odd dimensions
        self.register_buffer("pe", pe.unsqueeze(0))                        # fixed, not trained

    def forward(self, x):                                                  # x: (batch, seq, d_model)
        return self.dropout(x + self.pe[:, : x.size(1)])

## 2. Scaled dot-product attention

In [3]:
# 2. Scaled dot-product attention -------------------------------------------------------
def scaled_dot_product_attention(Q, K, V, mask=None, dropout=None):
    """softmax(Q K^T / sqrt(d_k)) V. mask: 1 = may attend, 0 = blocked."""
    scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.size(-1))
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)   # -1e9, not -inf: a fully masked row stays finite
    weights = F.softmax(scores, dim=-1)
    if dropout is not None:
        weights = dropout(weights)                     # randomly drop some attention links in training
    return weights @ V, weights

## 3. Multi-head attention

In [4]:
# 3. Multi-head attention ---------------------------------------------------------------
class MultiHeadAttention(nn.Module):
    """Project, split into heads, attend in parallel, concatenate, project back."""
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.h, self.d_k = num_heads, d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.attn = None                                                   # last weights, for inspection

    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        split = lambda x: x.view(B, -1, self.h, self.d_k).transpose(1, 2)  # (B, h, seq, d_k)
        Q, K, V = split(self.W_Q(query)), split(self.W_K(key)), split(self.W_V(value))
        if mask is not None:
            mask = mask.unsqueeze(1)                                       # same mask for every head
        out, self.attn = scaled_dot_product_attention(Q, K, V, mask, self.dropout)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.h * self.d_k)  # concat heads
        return self.W_O(out)

## 4. Position-wise feed-forward network

In [5]:
# 4. Position-wise feed-forward network -------------------------------------------------
class PositionwiseFeedForward(nn.Module):
    """Linear(d_model -> d_ff) -> ReLU -> Dropout -> Linear(d_ff -> d_model), per position."""
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1, self.linear2 = nn.Linear(d_model, d_ff), nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

## 5. Add & Norm

In [6]:
# 5. Add & Norm -------------------------------------------------------------------------
class LayerNorm(nn.Module):
    """gamma * (x - mean) / sqrt(var + eps) + beta, over the feature dimension."""
    def __init__(self, features, eps=1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(features))
        self.beta = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        return self.gamma * (x - mean) / torch.sqrt(var + self.eps) + self.beta


class SublayerConnection(nn.Module):
    """Post-LN, as in the original paper: LayerNorm(x + Dropout(sublayer(x)))."""
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm, self.dropout = LayerNorm(d_model), nn.Dropout(dropout)

    def forward(self, x, sublayer):
        return self.norm(x + self.dropout(sublayer(x)))

## 6. Encoder and decoder layers

In [7]:
# 6. Encoder and decoder layers --------------------------------------------------------
class EncoderLayer(nn.Module):
    """Self-attention -> Add & Norm -> FFN -> Add & Norm."""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.sub1, self.sub2 = SublayerConnection(d_model, dropout), SublayerConnection(d_model, dropout)

    def forward(self, x, src_mask=None):
        x = self.sub1(x, lambda x: self.self_attn(x, x, x, src_mask))     # Q = K = V = x
        return self.sub2(x, self.ffn)


class DecoderLayer(nn.Module):
    """Masked self-attention -> Add & Norm -> cross-attention -> Add & Norm -> FFN -> Add & Norm."""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.sub1, self.sub2, self.sub3 = (SublayerConnection(d_model, dropout) for _ in range(3))

    def forward(self, x, memory, src_mask=None, tgt_mask=None):
        x = self.sub1(x, lambda x: self.self_attn(x, x, x, tgt_mask))     # look back only
        x = self.sub2(x, lambda x: self.cross_attn(x, memory, memory, src_mask))  # Q: decoder, K/V: encoder
        return self.sub3(x, self.ffn)

## 7. The stacks

In [8]:
# 7. Stacks -----------------------------------------------------------------------------
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout=0.1, max_len=5000):
        super().__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model, dropout, max_len)
        self.layers = nn.ModuleList(EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers))

    def forward(self, src, src_mask=None):
        x = self.pos(self.embed(src) * math.sqrt(self.d_model))            # scale, then add positions
        for layer in self.layers:
            x = layer(x, src_mask)
        return x


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout=0.1, max_len=5000):
        super().__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model, dropout, max_len)
        self.layers = nn.ModuleList(DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers))

    def forward(self, tgt, memory, src_mask=None, tgt_mask=None):
        x = self.pos(self.embed(tgt) * math.sqrt(self.d_model))
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return x

## 8. The full model, masks and greedy decoding

In [9]:
# 8. The full model, masks and greedy decoding -----------------------------------------
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=512, num_heads=8, d_ff=2048,
                 num_layers=6, dropout=0.1, max_len=5000):
        super().__init__()
        self.encoder = Encoder(src_vocab, d_model, num_heads, d_ff, num_layers, dropout, max_len)
        self.decoder = Decoder(tgt_vocab, d_model, num_heads, d_ff, num_layers, dropout, max_len)
        self.generator = nn.Linear(d_model, tgt_vocab)                     # logits over the vocabulary
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src, src_mask=None):
        return self.encoder(src, src_mask)

    def decode(self, tgt, memory, src_mask=None, tgt_mask=None):
        return self.generator(self.decoder(tgt, memory, src_mask, tgt_mask))

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        return self.decode(tgt, self.encode(src, src_mask), src_mask, tgt_mask)


def padding_mask(seq, pad_idx=0):                   # (B, 1, len): hide padding keys
    return (seq != pad_idx).unsqueeze(1)

def causal_mask(size, device=None):                 # (1, size, size): lower triangle
    return torch.tril(torch.ones(size, size, dtype=torch.bool, device=device)).unsqueeze(0)

def make_masks(src, tgt_in, pad_idx=0):
    return padding_mask(src, pad_idx), padding_mask(tgt_in, pad_idx) & causal_mask(tgt_in.size(1), tgt_in.device)

@torch.no_grad()
def greedy_decode(model, src, max_len, bos, eos, pad_idx=0):
    model.eval()
    src_mask = padding_mask(src, pad_idx)
    memory = model.encode(src, src_mask)
    ys = torch.full((src.size(0), 1), bos, dtype=torch.long, device=src.device)
    for _ in range(max_len):
        logits = model.decode(ys, memory, src_mask, causal_mask(ys.size(1), src.device))
        next_tok = logits[:, -1].argmax(-1, keepdim=True)                 # most likely next token
        ys = torch.cat([ys, next_tok], dim=1)
        if (next_tok == eos).all():
            break
    return ys

## 9. Checks: positional encoding, LayerNorm and attention against PyTorch, and the size of the base model

In [10]:
torch.manual_seed(0)

# A. Positional encoding
pe = PositionalEncoding(512, dropout=0.0).pe[0]
print("pos 1, dims 0-3:", [round(v, 4) for v in pe[1, :4].tolist()])
cos = torch.cosine_similarity
print(f"cos(PE0, PE1) = {cos(pe[0], pe[1], dim=0):.4f}   cos(PE0, PE10) = {cos(pe[0], pe[10], dim=0):.4f}")
for d in [0, 128, 256, 510]:
    print(f"dim {d:3d}: period = {2 * math.pi * 10000 ** (d / 512):,.1f} positions")

# B. LayerNorm and multi-head attention against PyTorch's own modules
x = torch.randn(2, 13, 64)
ln = LayerNorm(64)
print(f"max |LayerNorm - nn.LayerNorm| = {(ln(x) - nn.LayerNorm(64)(x)).abs().max():.1e}")
mha = MultiHeadAttention(64, 4, dropout=0.0)
ref = nn.MultiheadAttention(64, 4, dropout=0.0, batch_first=True)
with torch.no_grad():
    ref.in_proj_weight.copy_(torch.cat([mha.W_Q.weight, mha.W_K.weight, mha.W_V.weight]))
    ref.in_proj_bias.copy_(torch.cat([mha.W_Q.bias, mha.W_K.bias, mha.W_V.bias]))
    ref.out_proj.weight.copy_(mha.W_O.weight); ref.out_proj.bias.copy_(mha.W_O.bias)
    causal = causal_mask(13)
    ours = mha(x, x, x, causal.expand(2, -1, -1))
    theirs = ref(x, x, x, attn_mask=~causal[0])[0]
print(f"max |ours - nn.MultiheadAttention| (causal) = {(ours - theirs).abs().max():.1e}")

# C. The base model from the paper
model = Transformer(src_vocab=10000, tgt_vocab=8000)
n = lambda m: sum(p.numel() for p in m.parameters())
print(f"parameters: total {n(model):,} | one encoder layer {n(model.encoder.layers[0]):,} | "
      f"one decoder layer {n(model.decoder.layers[0]):,}")
src, tgt = torch.randint(3, 10000, (2, 10)), torch.randint(3, 8000, (2, 12))
src_mask, tgt_mask = make_masks(src, tgt)
print("src_mask", tuple(src_mask.shape), "tgt_mask", tuple(tgt_mask.shape),
      "logits", tuple(model.eval()(src, tgt, src_mask, tgt_mask).shape))

pos 1, dims 0-3: [0.8415, 0.5403, 0.8219, 0.5697]
cos(PE0, PE1) = 0.9731   cos(PE0, PE10) = 0.6789
dim   0: period = 6.3 positions
dim 128: period = 62.8 positions
dim 256: period = 628.3 positions
dim 510: period = 60,611.5 positions
max |LayerNorm - nn.LayerNorm| = 2.4e-07
max |ours - nn.MultiheadAttention| (causal) = 0.0e+00


parameters: total 57,458,496 | one encoder layer 3,152,384 | one decoder layer 4,204,032
src_mask (2, 1, 10) tgt_mask (2, 12, 12) logits (2, 12, 8000)


## 10. Train it: reverse amino-acid sequences

A small model (2 layers, d_model = 64, 4 heads) learns to reverse random amino-acid strings of 8–16 residues, with the paper's warm-up schedule, label smoothing and gradient clipping. Then it reads the CDR3β `CASSIRSSYEQYF` backwards.

In [11]:
import time
torch.manual_seed(0)

AA = "ACDEFGHIKLMNPQRSTVWY"
PAD, BOS, EOS = 0, 1, 2
stoi = {a: i + 3 for i, a in enumerate(AA)}
itos = {i: a for a, i in stoi.items()}
V = len(AA) + 3

def batch(n, lo=8, hi=16):
    """Random amino-acid strings; the target is the same string reversed."""
    lens = torch.randint(lo, hi + 1, (n,))
    src = torch.full((n, hi), PAD); tgt = torch.full((n, hi + 2), PAD)
    for b, L in enumerate(lens):
        s = torch.randint(3, V, (L,))
        src[b, :L] = s
        tgt[b, 0], tgt[b, 1:L + 1], tgt[b, L + 1] = BOS, s.flip(0), EOS
    return src, tgt

model = Transformer(V, V, d_model=64, num_heads=4, d_ff=256, num_layers=2, dropout=0.1)
opt = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
warmup = 400
sched = torch.optim.lr_scheduler.LambdaLR(           # the paper's schedule: warm up, then 1/sqrt(step)
    opt, lambda s: 64 ** -0.5 * min((s + 1) ** -0.5, (s + 1) * warmup ** -1.5))
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.1)

t0 = time.time()
for step in range(1, 3001):
    model.train()
    src, tgt = batch(128)
    tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]        # "shifted right": predict token t+1 from tokens <= t
    src_mask, tgt_mask = make_masks(src, tgt_in)
    logits = model(src, tgt_in, src_mask, tgt_mask)
    loss = loss_fn(logits.reshape(-1, V), tgt_out.reshape(-1))
    opt.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); sched.step()
    if step % 500 == 0:
        print(f"step {step:4d}  loss {loss.item():.3f}")
print(f"training time: {time.time() - t0:.0f} s")

src, tgt = batch(1000)
out = greedy_decode(model, src, max_len=18, bos=BOS, eos=EOS)
ok = sum(out[b, 1:len(tgt[b].nonzero())].tolist() == tgt[b, 1:len(tgt[b].nonzero())].tolist() for b in range(1000))
print(f"held-out sequences reversed exactly: {ok}/1000")

cdr3 = "CASSIRSSYEQYF"
src = torch.tensor([[stoi[a] for a in cdr3]])
out = greedy_decode(model, src, max_len=18, bos=BOS, eos=EOS)[0, 1:].tolist()
print("input :", cdr3)
print("output:", "".join(itos[i] for i in out if i not in (EOS, PAD)))

step  500  loss 0.905


step 1000  loss 0.697


step 1500  loss 0.670


step 2000  loss 0.660


step 2500  loss 0.639


step 3000  loss 0.627
training time: 250 s


held-out sequences reversed exactly: 999/1000
input : CASSIRSSYEQYF
output: FYQEYSSRISSAC
